# Financial Fundamentals Analysis — Run on Colab GPU

This notebook runs the full pipeline (parsing, embeddings, NER, sentiment, KPI extraction, RAG) on a Colab GPU instead of your laptop's CPU.

**Before you start:** set the runtime to a GPU. Go to **Runtime → Change runtime type → Hardware accelerator → GPU** (T4 is fine), then Save. The code auto-detects the GPU (`torch.cuda.is_available()`) — you don't need to pass any flags either way, but it only helps if a GPU runtime is actually selected.

For the full reference docs (config options, output schema, troubleshooting), see [README.md](https://github.com/abhinaba01/fundamental-financial-analysis/blob/main/README.md) and [RUNNING.md](https://github.com/abhinaba01/fundamental-financial-analysis/blob/main/RUNNING.md) in the repo.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi

If this errors with "command not found" or shows no GPU, go back to **Runtime → Change runtime type** and select a GPU, then re-run this cell.

## 2. Clone the repository

In [ ]:
!git clone https://github.com/abhinaba01/fundamental-financial-analysis.git
%cd fundamental-financial-analysis

## 3. Install dependencies

Colab's runtime already ships a CUDA-enabled `torch`, so this won't reinstall it (the `>=2.0.0` constraint in `requirements.txt` is already satisfied) — it just adds everything else. Expect this to take a few minutes.

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm

## 4. Set your OpenAI API key (optional)

This is only needed for the RAG agent's `gpt-4o` chain-of-thought generation. Without it, the pipeline still runs end-to-end — NER, sentiment, KPI extraction, and embeddings are all local — but the RAG answer falls back to extractive synthesis (stitched-together chunk text) instead of an LLM-generated answer.

**Recommended**: use Colab's built-in Secrets manager instead of typing your key into a cell. Click the key icon (🔑) in the left sidebar, add a secret named `OPENAI_API_KEY`, and grant this notebook access. The cell below picks it up automatically; if it's not set, it falls back to an interactive prompt (the key won't be echoed or saved in the notebook).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded OPENAI_API_KEY from Colab Secrets.")
except Exception:
    from getpass import getpass
    key = getpass("OPENAI_API_KEY (leave blank to skip and use extractive fallback): ")
    if key:
        os.environ["OPENAI_API_KEY"] = key
    else:
        print("No key set - RAG will use extractive synthesis instead of gpt-4o.")

## 5. Confirm the code will actually use the GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

`src/main.py` auto-detects this at runtime and puts embeddings, NER, and sentiment models on `cuda` automatically — no `--gpu` flag needed. `--cpu` / `--gpu` are still there if you ever want to force one explicitly.

## 6. Run the pipeline on the bundled sample document

First run downloads the models (`BAAI/bge-large-en-v1.5`, `dslim/bert-base-NER`, `ProsusAI/finbert`) - a couple of minutes even on a fast connection. Every run after that is quick.

In [ ]:
!python -m src.main \
  --document test_small.txt \
  --query "What is the revenue and gross margin?" \
  --output report.json

## 7. View the report

In [ ]:
import json

with open("report.json", encoding="utf-8") as f:
    report = json.load(f)

print("Summary:", report.get("summary"), "\n")
print(json.dumps(report, indent=2))

## 8. (Optional) Analyze your own document

Upload a PDF, TXT, HTML, or JSON financial document from your computer, then point the CLI at it.

In [ ]:
from google.colab import files

uploaded = files.upload()
document_path = next(iter(uploaded))
print("Uploaded:", document_path)

In [ ]:
query = "What are the primary risk factors?"  # edit this

!python -m src.main --document "{document_path}" --query "{query}" --output my_report.json

## 9. (Optional) Download the report to your computer

In [ ]:
from google.colab import files

files.download("my_report.json")  # or "report.json" for the sample-document run

## Notes

- **Colab sessions are ephemeral.** Everything here (cloned repo, downloaded models, `data/vector_store/`) disappears when the runtime disconnects or recycles. If you want the vector store to persist across sessions, mount Google Drive and point `EmbeddingPipeline`'s `vector_store_path` there instead — not set up in this notebook by default, since it adds a permissions prompt most people don't need for a one-off analysis.
- **Free-tier Colab GPUs get disconnected/recycled** after periods of inactivity or after a few hours. A paid tier (which you have) is more stable but not unlimited — don't leave this idle mid-run.
- **Re-running cell 6/the upload cell** with the same document is safe: chunks are upserted into ChromaDB by ID, not appended, so nothing duplicates.
- To pull in code changes after the repo is updated on GitHub, re-run `!git -C fundamental-financial-analysis pull` rather than re-cloning.